# Article 05 — Automated FAIR Score Calculation with Python
## Building Towards an Agentic FAIR Assessor

**Series:** FAIR Data Maturity Framework  
**Author:** Ali Shahmohammadi, Ph.D. — Takeda Pharmaceutical  
**Prerequisites:** Articles 01–04

---

## Overview

This article demonstrates how to go beyond manual assessment and begin **automating**  
portions of the FAIR evaluation. We will:

1. Query a public API and extract metadata programmatically
2. Apply the RDA indicators to that metadata automatically where possible
3. Generate a FAIR score from API-derived evidence
4. Lay the groundwork for the **Agentic FAIR Scorer** (next phase of this series)

> **Note:** Full end-to-end automated scoring is the subject of the next module in this series:  
> `fair_agent/` — an agentic AI system using LangGraph + GPT to score FAIR compliance  
> given any database API endpoint. That module is currently under development.

In [ ]:
# ── Check what can be assessed automatically from a public API ─────────────
# We'll use Zenodo's public REST API to assess a published dataset

import httpx, json
from rich.console import Console
from rich.json import JSON

console = Console()

# Query a published Zenodo record (FAIRsharing dataset — example)
record_id = "7654321"
url = f"https://zenodo.org/api/records/{record_id}"

try:
    resp = httpx.get(url, timeout=10)
    if resp.status_code == 200:
        data = resp.json()
        console.print("[green]✓ Zenodo API returned metadata[/green]")
        console.print(f"  Title: {data.get('metadata', {}).get('title', 'N/A')}")
        console.print(f"  DOI: {data.get('doi', 'N/A')}")
        console.print(f"  Files: {len(data.get('files', []))}")
    else:
        console.print(f"[yellow]Note: Record {record_id} returned {resp.status_code}[/yellow]")
        console.print("Using a mock metadata response for demonstration...")
except Exception as e:
    console.print(f"[yellow]API not reachable: {e}[/yellow]")
    console.print("Using a mock metadata response for demonstration...")

In [ ]:
# ── Mock a rich metadata API response for demonstration ───────────────────
MOCK_ZENODO_RESPONSE = {
    "id": "7654321",
    "doi": "10.5281/zenodo.7654321",
    "metadata": {
        "title": "Transcriptomic profiling of CAR-T cell manufacturing — batch comparison",
        "creators": [{"name": "Shahmohammadi, Ali", "orcid": "0000-0002-XXXX-XXXX"}],
        "description": "RNA-seq data from CAR-T manufacturing process development. Three batches, 12 timepoints.",
        "keywords": ["CAR-T", "RNA-seq", "cell therapy", "manufacturing"],
        "license": {"id": "cc-by-4.0"},
        "access_right": "open",
        "resource_type": {"type": "dataset"},
        "related_identifiers": [
            {"identifier": "10.1000/protocol.xyz", "relation": "isCompiledBy", "scheme": "doi"},
            {"identifier": "10.1038/s41587-023-0001-1", "relation": "isSupplementTo", "scheme": "doi"},
        ],
        "dates": [{"date": "2024-03-15", "type": "Created"}],
        "language": "eng",
    },
    "files": [
        {"key": "counts_matrix.csv", "size": 15_000_000},
        {"key": "sample_metadata.tsv", "size": 45_000},
        {"key": "README.md", "size": 5_000},
    ],
    "links": {
        "self": "https://zenodo.org/api/records/7654321",
        "doi": "https://doi.org/10.5281/zenodo.7654321",
    },
}

print("Mock Zenodo response loaded.")
print(f"  DOI: {MOCK_ZENODO_RESPONSE['doi']}")
print(f"  Title: {MOCK_ZENODO_RESPONSE['metadata']['title']}")
print(f"  Files: {[f['key'] for f in MOCK_ZENODO_RESPONSE['files']]}")
print(f"  Licence: {MOCK_ZENODO_RESPONSE['metadata']['license']['id']}")
print(f"  Related IDs: {len(MOCK_ZENODO_RESPONSE['metadata']['related_identifiers'])}")

In [ ]:
# ── Semi-automated FAIR assessment from API metadata ──────────────────────
from fair_toolkit import ManualFAIRAssessor, ComplianceScore

def auto_assess_from_zenodo(record: dict, dataset_id: str, title: str) -> ManualFAIRAssessor:
    """
    Automatically score indicators that can be determined from Zenodo metadata.
    Returns an assessor with auto-scored indicators; remaining indicators = NOT_ASSESSED.
    This is the seed of the agentic FAIR scorer.
    """
    assessor = ManualFAIRAssessor(
        dataset_id=dataset_id,
        dataset_title=title,
        assessed_by="Semi-automated (Zenodo API)",
    )
    meta = record.get("metadata", {})

    # F1: PID check
    doi = record.get("doi", "")
    has_doi = bool(doi and doi.startswith("10."))
    assessor.score("RDA-F1-01D", ComplianceScore.COMPLIANT if has_doi else ComplianceScore.NOT_COMPLIANT,
                   evidence=f"DOI: {doi}" if has_doi else "No DOI found")
    assessor.score("RDA-F1-02D", ComplianceScore.COMPLIANT if has_doi else ComplianceScore.NOT_COMPLIANT,
                   evidence="DOI is globally unique by definition")
    assessor.score("RDA-F1-01M", ComplianceScore.COMPLIANT if has_doi else ComplianceScore.NOT_COMPLIANT,
                   evidence=f"Zenodo metadata record has DOI: {doi}")
    assessor.score("RDA-F1-02M", ComplianceScore.COMPLIANT if has_doi else ComplianceScore.NOT_COMPLIANT)

    # F2: Rich metadata — check for key fields
    has_desc = bool(meta.get("description", "").strip())
    has_keywords = bool(meta.get("keywords"))
    has_creators = bool(meta.get("creators"))
    richness = sum([has_desc, has_keywords, has_creators])
    f2_score = {3: ComplianceScore.COMPLIANT, 2: ComplianceScore.PARTIALLY_COMPLIANT}.get(
        richness, ComplianceScore.NOT_COMPLIANT)
    assessor.score("RDA-F2-01M", f2_score,
                   evidence=f"Description: {has_desc}, Keywords: {has_keywords}, Creators: {has_creators}")

    # F3: Metadata → data link
    links = record.get("links", {})
    has_doi_link = "doi" in links
    assessor.score("RDA-F3-01M",
                   ComplianceScore.COMPLIANT if has_doi_link else ComplianceScore.PARTIALLY_COMPLIANT,
                   evidence="Zenodo metadata record resolves to data via DOI link")

    # A1.1: Open protocol (Zenodo uses HTTPS — open)
    assessor.score("RDA-A1.1-01M", ComplianceScore.COMPLIANT, evidence="Zenodo uses HTTPS — open, free protocol")
    assessor.score("RDA-A1.1-01D", ComplianceScore.COMPLIANT, evidence="Files downloadable via HTTPS without auth")
    assessor.score("RDA-A1-02M", ComplianceScore.COMPLIANT, evidence="Zenodo REST API is free and documented")
    assessor.score("RDA-A1-04M", ComplianceScore.COMPLIANT, evidence="Zenodo metadata accessible at api/records/{id}")

    # A2: Metadata persistence — Zenodo/DataCite guarantees this
    assessor.score("RDA-A2-01M", ComplianceScore.COMPLIANT,
                   evidence="DataCite provides permanent metadata retention even if Zenodo record is deleted")

    # R1.1: Licence
    licence = meta.get("license", {}).get("id", "")
    has_std_licence = licence in ("cc-by-4.0", "cc0-1.0", "cc-by-sa-4.0", "mit", "apache-2.0")
    assessor.score("RDA-R1.1-01M",
                   ComplianceScore.COMPLIANT if licence else ComplianceScore.NOT_COMPLIANT,
                   evidence=f"Licence: {licence}")
    assessor.score("RDA-R1.1-02M",
                   ComplianceScore.COMPLIANT if has_std_licence else ComplianceScore.PARTIALLY_COMPLIANT,
                   evidence=f"Licence ID '{licence}' is a standard SPDX identifier")
    assessor.score("RDA-R1.1-03M",
                   ComplianceScore.COMPLIANT if has_std_licence else ComplianceScore.NOT_COMPLIANT,
                   evidence="SPDX licence ID is machine-readable")

    # I3: Qualified references
    related = meta.get("related_identifiers", [])
    has_typed_refs = any(r.get("relation") for r in related)
    assessor.score("RDA-I3-01M",
                   ComplianceScore.COMPLIANT if has_typed_refs else ComplianceScore.NOT_COMPLIANT,
                   evidence=f"Found {len(related)} typed relatedIdentifiers" if has_typed_refs else "No typed references")

    return assessor

assessor = auto_assess_from_zenodo(
    MOCK_ZENODO_RESPONSE,
    dataset_id="ZENODO-7654321",
    title=MOCK_ZENODO_RESPONSE["metadata"]["title"],
)

print(f"Auto-assessment complete. Progress: {assessor.progress()}")
assessor.print_scorecard()

## The Path to Full Agentic FAIR Scoring

The semi-automated approach above handles **about 15–20 of the 41 indicators**  
automatically from structured API metadata.

The remaining indicators require:
- **Natural language understanding** (e.g., checking if a description mentions data quality)
- **Format validation** (e.g., validating a CSV against a schema)
- **External lookups** (e.g., checking if a vocabulary term is in an OBO ontology)
- **Domain knowledge** (e.g., determining if a community standard is appropriate)

This is exactly where an **Agentic AI system** excels.

### The Fair-Agent Architecture (Next Module)

```
Input: any API endpoint + dataset identifier
         │
         ▼
┌─────────────────────────────────────────────────────┐
│              FAIRAssessorAgent (LangGraph)           │
│                                                     │
│  Node 1: MetadataFetcherAgent                       │
│    → Calls API, extracts structured metadata        │
│                                                     │
│  Node 2: StructuralAssessorAgent                    │
│    → Auto-scores ~20 structural indicators          │
│      (PID presence, licence, protocol type, etc.)   │
│                                                     │
│  Node 3: ContentAnalysisAgent (GPT)                 │
│    → Reads description/metadata text                │
│    → Scores semantic indicators:                    │
│      richness, provenance language, quality flags   │
│                                                     │
│  Node 4: ExternalVerifierAgent                      │
│    → Calls BioPortal/OLS to verify vocabulary PIDs  │
│    → Calls FAIRsharing to verify standards          │
│    → Validates file formats                         │
│                                                     │
│  Node 5: GovernanceReporterAgent                    │
│    → Compiles FAIRAssessmentResult                  │
│    → Generates gap analysis and remediation plan    │
│    → Routes low-scoring indicators for human review │
└─────────────────────────────────────────────────────┘
         │
         ▼
Output: FAIRAssessmentResult with 41 scored indicators,
        FAIR score by dimension, and remediation roadmap
```

This is the target architecture for the `fair_agent/` module currently in development.

In [ ]:
# Preview: what the agentic interface will look like
# (This is a design sketch — the full implementation is in the next module)

# from fair_agent import FAIRAssessorAgent  # coming in next module
# 
# agent = FAIRAssessorAgent.from_env(model="gpt-5.2")
# 
# result = agent.assess(
#     api_endpoint="https://zenodo.org/api/records/7654321",
#     dataset_id="ZENODO-7654321",
# )
# 
# print(f"Overall FAIR score: {result.overall_score}%")
# result.print_scorecard()
# result.save("zenodo_7654321_fair_assessment.json")

print("Agentic FAIR Scorer — fair_agent/ module — coming in the next series article.")
print()
print("Capabilities planned:")
capabilities = [
    "Accept any REST API endpoint as input",
    "Fetch and parse metadata in JSON, JSON-LD, XML, DCAT formats",
    "Auto-score ~20 structural indicators from metadata fields",
    "Use GPT to assess 15 semantic/content-based indicators",
    "Query BioPortal/OLS to verify ontology term usage (I2 indicators)",
    "Validate file formats against community standards (R1.3 indicators)",
    "Generate remediation roadmap prioritised by Essential → Important → Useful",
    "Export as JSON, CSV, or push to enterprise data catalogue",
]
for c in capabilities:
    print(f"  ✓ {c}")